#  FAISS Tuning Notebook

In [ ]:
# ==== Config + env ====
import os
from pathlib import Path
from urllib.parse import urlsplit

from dotenv import load_dotenv

ENV_FILE = '.env'   # change to '.env.local' if needed
EMBED_LIMIT = 5000
DATASET_FILTER = None   # e.g. 'kaggle'
RANDOM_SEED = 42
MAX_DOWNLOAD_WORKERS = int(os.getenv('EMBED_DOWNLOAD_WORKERS', '16'))
DOWNLOAD_RETRIES = int(os.getenv('EMBED_DOWNLOAD_RETRIES', '2'))
EMBED_MODEL = os.getenv('EMBED_MODEL', 'google/siglip2-base-patch16-naflex')
EMBED_DEVICE = os.getenv('EMBED_DEVICE', 'cpu')


def find_backend_api_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if p.name == 'backend-api':
            return p
        candidate = p / 'backend-api'
        if candidate.exists() and candidate.is_dir():
            return candidate
    raise RuntimeError('Could not locate backend-api root')


def redact_dsn(dsn: str) -> str:
    try:
        parts = urlsplit(dsn)
        host = parts.hostname or 'unknown'
        port = f':{parts.port}' if parts.port else ''
        path = parts.path or ''
        return f'{parts.scheme}://***:***@{host}{port}{path}'
    except Exception:
        return '***'


def redact_url(url: str | None) -> str:
    if not url:
        return 'missing'
    try:
        parts = urlsplit(url)
        host = parts.hostname or 'unknown'
        if len(host) > 18:
            host = host[:8] + '...' + host[-8:]
        return f'{parts.scheme}://{host}'
    except Exception:
        return '***'


BACKEND_API_ROOT = find_backend_api_root(Path.cwd())
ENV_PATH = BACKEND_API_ROOT / ENV_FILE
load_dotenv(ENV_PATH, override=False)

DB_DSN = os.getenv('DB_DSN') or os.getenv('DB_URL')
if not DB_DSN:
    raise RuntimeError('Missing DB_DSN/DB_URL in env')

S3_ENDPOINT_URL = os.getenv('S3_ENDPOINT_URL')
S3_REGION = os.getenv('S3_REGION', 'us-east-1')
S3_ACCESS_KEY_ID = os.getenv('S3_ACCESS_KEY_ID')
S3_SECRET_ACCESS_KEY = os.getenv('S3_SECRET_ACCESS_KEY')
S3_BUCKET = os.getenv('S3_BUCKET')
S3_FORCE_PATH_STYLE = str(os.getenv('S3_FORCE_PATH_STYLE', 'true')).lower() in {'1', 'true', 'yes', 'on'}

if not all([S3_ENDPOINT_URL, S3_ACCESS_KEY_ID, S3_SECRET_ACCESS_KEY, S3_BUCKET]):
    raise RuntimeError('Missing S3 env vars (endpoint/key/secret/bucket)')

print('env file:', ENV_PATH)
print('db dsn:', redact_dsn(DB_DSN))
print('s3 endpoint:', redact_url(S3_ENDPOINT_URL))
print('s3 bucket:', S3_BUCKET)
print('download workers:', MAX_DOWNLOAD_WORKERS)
print('download retries:', DOWNLOAD_RETRIES)
print('embed model:', EMBED_MODEL)
print('embed device:', EMBED_DEVICE)


In [ ]:
import sys

print('Installing notebook dependencies into kernel env:', sys.executable)
!uv pip install --python "{sys.executable}" faiss-cpu numpy pandas matplotlib pillow boto3 psycopg2-binary python-dotenv tqdm transformers torch


In [ ]:
# ==== Load embeddings + metadata from Postgres/S3 (concurrent downloads) ====
import hashlib
import io
import json
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import UTC, datetime

import boto3
from botocore.exceptions import ClientError
import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg2
from PIL import Image
from tqdm.auto import tqdm

plt.rcParams['figure.figsize'] = (14, 8)
pd.set_option('display.max_colwidth', 140)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

conn = psycopg2.connect(DB_DSN)
conn.autocommit = False


def make_s3_client():
    return boto3.client(
        's3',
        endpoint_url=S3_ENDPOINT_URL,
        region_name=S3_REGION,
        aws_access_key_id=S3_ACCESS_KEY_ID,
        aws_secret_access_key=S3_SECRET_ACCESS_KEY,
        config=boto3.session.Config(s3={'addressing_style': 'path' if S3_FORCE_PATH_STYLE else 'auto'}),
    )


s3 = make_s3_client()

EMBEDDING_FILE_CACHE_DIR = BACKEND_API_ROOT / 'data' / 'cache' / 'embeddings'
EMBEDDING_FILE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print('embedding cache dir:', EMBEDDING_FILE_CACHE_DIR)


def _cache_path_for_s3_key(key: str) -> Path:
    digest = hashlib.sha1(key.encode('utf-8')).hexdigest()
    return EMBEDDING_FILE_CACHE_DIR / f'{digest}.npy'


def load_embedding_from_s3(client, key: str) -> np.ndarray:
    cache_path = _cache_path_for_s3_key(key)
    if cache_path.exists():
        return np.load(cache_path).astype(np.float32).reshape(-1)

    obj = client.get_object(Bucket=S3_BUCKET, Key=key)
    body = obj['Body'].read()
    vec = np.load(io.BytesIO(body)).astype(np.float32).reshape(-1)

    tmp_path = cache_path.with_suffix('.tmp')
    with open(tmp_path, 'wb') as f:
        np.save(f, vec)
    tmp_path.replace(cache_path)
    return vec


def to_text_key(image_embed_key: str) -> str:
    return f"{image_embed_key[:-4]}_text.npy" if image_embed_key.endswith('.npy') else f'{image_embed_key}_text.npy'


def fetch_one(index_and_row):
    idx, row = index_and_row
    key = row['embed_s3_key']
    text_key = to_text_key(key)
    client = make_s3_client()

    last_exc = None
    for _ in range(max(1, DOWNLOAD_RETRIES + 1)):
        try:
            img_emb = load_embedding_from_s3(client, key)

            txt_emb = None
            try:
                txt_emb = load_embedding_from_s3(client, text_key)
            except ClientError:
                txt_emb = None
            except Exception:
                txt_emb = None

            return idx, row, img_emb, txt_emb, None
        except Exception as exc:
            last_exc = exc

    return idx, row, None, None, str(last_exc)


def load_data(limit: int | None = None, dataset: str | None = None):
    sql = '''
        SELECT
            i.id AS image_id,
            i.sha256,
            i.s3_key,
            i.dataset,
            i.width,
            i.height,
            p.embed_s3_key,
            a.caption_text,
            a.ocr_text
        FROM images i
        JOIN processing p ON p.image_id = i.id
        LEFT JOIN annotations a ON a.image_id = i.id
        WHERE p.embed_status = 'DONE'
          AND p.embed_s3_key IS NOT NULL
          {dataset_clause}
        ORDER BY i.id ASC
        {limit_clause}
    '''
    dataset_clause = "AND i.dataset = %(dataset)s" if dataset else ""
    limit_clause = f"LIMIT {int(limit)}" if limit else ""
    sql = sql.format(dataset_clause=dataset_clause, limit_clause=limit_clause)

    with conn.cursor() as cur:
        cur.execute(sql, {'dataset': dataset} if dataset else {})
        rows = cur.fetchall()

    cols = ['image_id', 'sha256', 's3_key', 'dataset', 'width', 'height', 'embed_s3_key', 'caption', 'ocr_text']
    raw_df = pd.DataFrame(rows, columns=cols)
    records = raw_df.to_dict(orient='records')

    ok = []
    failed = 0
    workers = max(1, min(MAX_DOWNLOAD_WORKERS, len(records) or 1))
    print(f'downloading {len(records)} embedding rows with workers={workers}...')

    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(fetch_one, x) for x in enumerate(records)]
        for fut in tqdm(as_completed(futs), total=len(futs), desc='Embedding downloads'):
            idx, row, img_emb, txt_emb, err = fut.result()
            if img_emb is None:
                failed += 1
                continue
            ok.append((idx, row, img_emb, txt_emb))

    if not ok:
        raise RuntimeError('No embeddings loaded from DB/S3')

    ok.sort(key=lambda x: x[0])
    keep_rows = [r for _, r, _, _ in ok]
    img_vectors = [e for _, _, e, _ in ok]

    X_image = np.stack(img_vectors).astype(np.float32)
    meta = pd.DataFrame(keep_rows).drop(columns=['embed_s3_key'])

    text_rows = []
    text_vectors = []
    text_dim = None
    text_dim_mismatch = 0

    for _, row, _, txt_emb in ok:
        if txt_emb is None:
            continue
        if text_dim is None:
            text_dim = int(txt_emb.shape[0])
        if int(txt_emb.shape[0]) != text_dim:
            text_dim_mismatch += 1
            continue
        text_rows.append(row)
        text_vectors.append(txt_emb)

    X_text = np.stack(text_vectors).astype(np.float32) if text_vectors else None
    text_meta = pd.DataFrame(text_rows).drop(columns=['embed_s3_key']) if text_rows else pd.DataFrame()

    print({
        'image_loaded': len(img_vectors),
        'text_loaded': len(text_vectors),
        'text_dim_mismatch': text_dim_mismatch,
        'failed': failed,
        'image_dimension': int(X_image.shape[1]),
    })
    return X_image, meta, X_text, text_meta


X, meta_df, X_text, text_meta_df = load_data(limit=EMBED_LIMIT, dataset=DATASET_FILTER)
print('image embeddings shape:', X.shape)
if X_text is not None:
    print('text embeddings shape:', X_text.shape)
else:
    print('text embeddings shape: none')
display(meta_df.head(5))
if not text_meta_df.empty:
    print('sample rows with text embeddings:')
    display(text_meta_df.head(5))


In [ ]:
# ==== FAISS helpers: build/search/save/list/load ====
INDEX_DIR = BACKEND_API_ROOT / 'data' / 'cache' / 'tuning_indexes'
INDEX_DIR.mkdir(parents=True, exist_ok=True)


def _local_versions_df() -> pd.DataFrame:
    out = []
    for d in sorted(INDEX_DIR.glob('*')):
        if not d.is_dir() or not (d / 'metadata.json').exists():
            continue
        m = json.loads((d / 'metadata.json').read_text())
        m['version'] = d.name
        m['has_text_index'] = bool(m.get('has_text_index', (d / 'text_index.faiss').exists()))
        out.append(m)
    return pd.DataFrame(out).sort_values('created_at', ascending=False) if out else pd.DataFrame()


def normalize(x: np.ndarray) -> np.ndarray:
    y = np.ascontiguousarray(x.astype(np.float32))
    faiss.normalize_L2(y)
    return y


def reciprocal_rank_fusion(
    image_results: list[tuple[int, float]],
    text_results: list[tuple[int, float]],
    k: int = 60,
) -> list[tuple[int, float]]:
    scores: dict[int, float] = {}
    for rank, (image_id, _) in enumerate(image_results, start=1):
        scores[image_id] = scores.get(image_id, 0.0) + 1.0 / (rank + k)
    for rank, (image_id, _) in enumerate(text_results, start=1):
        scores[image_id] = scores.get(image_id, 0.0) + 1.0 / (rank + k)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def build_index(x: np.ndarray, kind: str, **params):
    x = normalize(x)
    n, d = x.shape

    if kind == 'flat':
        idx = faiss.IndexFlatIP(d)
    elif kind == 'ivf':
        nlist = int(params.get('nlist', min(4096, max(64, int(np.sqrt(n))))))
        q = faiss.IndexFlatIP(d)
        idx = faiss.IndexIVFFlat(q, d, nlist, faiss.METRIC_INNER_PRODUCT)
    elif kind == 'hnsw':
        m = int(params.get('m', 32))
        idx = faiss.IndexHNSWFlat(d, m, faiss.METRIC_INNER_PRODUCT)
        idx.hnsw.efConstruction = int(params.get('efConstruction', 120))
    elif kind == 'ivfpq':
        nlist = int(params.get('nlist', min(4096, max(64, int(np.sqrt(n))))))
        m = int(params.get('m', 16))
        nbits = int(params.get('nbits', 8))
        q = faiss.IndexFlatIP(d)
        idx = faiss.IndexIVFPQ(q, d, nlist, m, nbits, faiss.METRIC_INNER_PRODUCT)
    else:
        raise ValueError(f'Unknown kind: {kind}')

    if not idx.is_trained:
        idx.train(x)
    idx.add(x)
    return idx


def set_search_params(idx, **params):
    ps = faiss.ParameterSpace()
    for k, v in params.items():
        try:
            ps.set_index_parameter(idx, str(k), float(v))
        except Exception:
            pass


def save_version(
    idx,
    image_ids: list[int],
    kind: str,
    build_params: dict,
    text_idx=None,
    text_image_ids: list[int] | None = None,
):
    version = f"{kind}-v{datetime.now(UTC).strftime('%Y%m%d-%H%M%S')}"
    vdir = INDEX_DIR / version
    vdir.mkdir(parents=True, exist_ok=True)

    faiss.write_index(idx, str(vdir / 'index.faiss'))
    (vdir / 'mapping.json').write_text(json.dumps({i: int(img_id) for i, img_id in enumerate(image_ids)}))
    (vdir / 'metadata.json').write_text(json.dumps({
        'version': version,
        'kind': kind,
        'build_params': build_params,
        'num_vectors': int(idx.ntotal),
        'dimension': int(idx.d),
        'has_text_index': text_idx is not None and text_image_ids is not None,
        'created_at': datetime.now(UTC).isoformat(),
    }, indent=2))

    if text_idx is not None and text_image_ids is not None:
        faiss.write_index(text_idx, str(vdir / 'text_index.faiss'))
        (vdir / 'text_mapping.json').write_text(json.dumps({i: int(img_id) for i, img_id in enumerate(text_image_ids)}))
        (vdir / 'text_metadata.json').write_text(json.dumps({
            'version': version,
            'kind': kind,
            'num_vectors': int(text_idx.ntotal),
            'dimension': int(text_idx.d),
            'created_at': datetime.now(UTC).isoformat(),
        }, indent=2))

    return version


def list_versions() -> pd.DataFrame:
    return _local_versions_df()


def load_version(version: str):
    vdir = INDEX_DIR / version
    idx = faiss.read_index(str(vdir / 'index.faiss'))
    mapping = {int(k): int(v) for k, v in json.loads((vdir / 'mapping.json').read_text()).items()}
    meta = json.loads((vdir / 'metadata.json').read_text())

    text_idx = None
    text_mapping = {}
    text_meta = {}
    if (vdir / 'text_index.faiss').exists() and (vdir / 'text_mapping.json').exists():
        text_idx = faiss.read_index(str(vdir / 'text_index.faiss'))
        text_mapping = {int(k): int(v) for k, v in json.loads((vdir / 'text_mapping.json').read_text()).items()}
        if (vdir / 'text_metadata.json').exists():
            text_meta = json.loads((vdir / 'text_metadata.json').read_text())

    return idx, mapping, meta, text_idx, text_mapping, text_meta


def search_faiss(idx, mapping: dict[int, int], query_vector: np.ndarray, k: int):
    q = query_vector.astype(np.float32)
    if q.ndim == 1:
        q = q.reshape(1, -1)
    faiss.normalize_L2(q)
    D, I = idx.search(q, k)
    out = []
    for pos, score in zip(I[0], D[0]):
        if pos < 0:
            continue
        img_id = mapping.get(int(pos))
        if img_id is not None:
            out.append((int(img_id), float(score)))
    return out

In [ ]:
# ==== Build and evaluate index families (flat/ivf/hnsw/ivfpq) ====
Xn = normalize(X)
image_ids = meta_df['image_id'].astype(int).tolist()

Xt = normalize(X_text) if X_text is not None else None
text_image_ids = text_meta_df['image_id'].astype(int).tolist() if not text_meta_df.empty else []

N, D = Xn.shape
print({'num_image_vectors': N, 'image_dim': D, 'num_text_vectors': int(Xt.shape[0]) if Xt is not None else 0})


if Xt is None or len(text_image_ids) == 0:
    raise RuntimeError('Hybrid evaluation requires text embeddings, but none were loaded.')


def max_reasonable_k(n_vectors: int) -> int:
    return max(1, n_vectors // 39)


def choose_nlist_candidates(n_vectors: int) -> list[int]:
    kmax = max_reasonable_k(n_vectors)
    base = [8, 16, 32, 64, 128, 256, 512]
    out = [k for k in base if k <= kmax]
    if not out:
        out = [max(1, min(8, n_vectors // 20))]
    return sorted(set(out))


def choose_pq_nbits(n_vectors: int) -> int:
    for b in [8, 7, 6, 5, 4]:
        if n_vectors >= 39 * (2 ** b):
            return b
    return 4


def row_to_results(I_row, D_row, mapping):
    out = []
    for pos, score in zip(I_row, D_row):
        if pos < 0:
            continue
        img_id = mapping.get(int(pos))
        if img_id is not None:
            out.append((int(img_id), float(score)))
    return out


def rank_of(ids: list[int], target: int) -> int | None:
    for i, image_id in enumerate(ids, start=1):
        if image_id == target:
            return i
    return None


nlist_candidates = choose_nlist_candidates(N)
pq_nbits = choose_pq_nbits(N)

if max(nlist_candidates) < 64:
    print(f"small dataset detected (N={N}): using nlist={nlist_candidates} and PQ nbits={pq_nbits}")

baseline_image = build_index(Xn, 'flat')
baseline_text = build_index(Xt, 'flat')

configs = [('flat', {})]
for nlist in nlist_candidates[-2:]:
    configs.append(('ivf', {'nlist': int(nlist)}))
configs.append(('hnsw', {'m': 32, 'efConstruction': 120}))
configs.append(('ivfpq', {'nlist': int(nlist_candidates[-1]), 'm': 16, 'nbits': int(pq_nbits)}))

qni = min(200, len(Xn))
qidx_img = np.random.choice(len(Xn), size=qni, replace=False)
Qi = Xn[qidx_img]

qnt = min(200, len(Xt))
qidx_txt = np.random.choice(len(Xt), size=qnt, replace=False)
Qt = Xt[qidx_txt]
qtrue_img_ids = [int(text_image_ids[i]) for i in qidx_txt]

K = 20
D_ref_img, I_ref_img = baseline_image.search(Qi, K)
D_ref_img_txt, I_ref_img_txt = baseline_image.search(Qt, K)
D_ref_txt, I_ref_txt = baseline_text.search(Qt, K)

ref_img_mapping = {i: int(img_id) for i, img_id in enumerate(image_ids)}
ref_txt_mapping = {i: int(img_id) for i, img_id in enumerate(text_image_ids)}
img_mapping = {i: int(img_id) for i, img_id in enumerate(image_ids)}
txt_mapping = {i: int(img_id) for i, img_id in enumerate(text_image_ids)}

eval_rows = []
saved = []

for kind, build_params in configs:
    image_idx = build_index(Xn, kind, **build_params)
    text_idx = build_index(Xt, kind, **build_params)

    param_sweep = [{}]
    if kind in {'ivf', 'ivfpq'}:
        nlist = int(build_params['nlist'])
        probes = [4, 8, 16, 32, 64]
        probes = [p for p in probes if p <= nlist]
        param_sweep = [{'nprobe': p} for p in probes] or [{'nprobe': 1}]
    elif kind == 'hnsw':
        param_sweep = [{'efSearch': 32}, {'efSearch': 64}, {'efSearch': 128}]

    for sp in param_sweep:
        set_search_params(image_idx, **sp)
        set_search_params(text_idx, **sp)

        t0 = time.perf_counter()
        D_img, I_img = image_idx.search(Qi, K)
        dt_img = (time.perf_counter() - t0) * 1000.0

        overlap_img = []
        for a, b in zip(I_ref_img, I_img):
            sa = set(int(x) for x in a if x >= 0)
            sb = set(int(x) for x in b if x >= 0)
            overlap_img.append(len(sa & sb) / max(1, len(sa)))

        t1 = time.perf_counter()
        D_txt, I_txt = text_idx.search(Qt, K)
        dt_txt = (time.perf_counter() - t1) * 1000.0

        overlap_txt = []
        for a, b in zip(I_ref_txt, I_txt):
            sa = set(int(x) for x in a if x >= 0)
            sb = set(int(x) for x in b if x >= 0)
            overlap_txt.append(len(sa & sb) / max(1, len(sa)))

        image_hits = 0
        hybrid_hits = 0
        image_rr = []
        hybrid_rr = []
        hybrid_overlap = []

        t2 = time.perf_counter()
        D_img_txt, I_img_txt = image_idx.search(Qt, K)
        dt_img_on_text = (time.perf_counter() - t2) * 1000.0

        for qi in range(len(Qt)):
            img_results = row_to_results(I_img_txt[qi], D_img_txt[qi], img_mapping)
            txt_results = row_to_results(I_txt[qi], D_txt[qi], txt_mapping)
            approx_hybrid = reciprocal_rank_fusion(img_results, txt_results, k=60)

            ref_img_results = row_to_results(I_ref_img_txt[qi], D_ref_img_txt[qi], ref_img_mapping)
            ref_txt_results = row_to_results(I_ref_txt[qi], D_ref_txt[qi], ref_txt_mapping)
            ref_hybrid = reciprocal_rank_fusion(ref_img_results, ref_txt_results, k=60)

            approx_hybrid_ids = [img_id for img_id, _ in approx_hybrid[:K]]
            ref_hybrid_ids = [img_id for img_id, _ in ref_hybrid[:K]]
            approx_image_ids = [img_id for img_id, _ in img_results[:K]]
            true_img_id = qtrue_img_ids[qi]

            if true_img_id in approx_image_ids:
                image_hits += 1
            if true_img_id in approx_hybrid_ids:
                hybrid_hits += 1

            ir = rank_of(approx_image_ids, true_img_id)
            hr = rank_of(approx_hybrid_ids, true_img_id)
            image_rr.append(0.0 if ir is None else 1.0 / ir)
            hybrid_rr.append(0.0 if hr is None else 1.0 / hr)

            sa = set(ref_hybrid_ids)
            sb = set(approx_hybrid_ids)
            hybrid_overlap.append(len(sa & sb) / max(1, len(sa)))

        hit_rate_image = image_hits / len(Qt)
        hit_rate_hybrid = hybrid_hits / len(Qt)

        eval_rows.append({
            'kind': kind,
            'build_params': json.dumps(build_params, sort_keys=True),
            'search_params': json.dumps(sp, sort_keys=True),
            'recall_at_k_image': round(float(np.mean(overlap_img)), 4),
            'recall_at_k_text': round(float(np.mean(overlap_txt)), 4),
            'recall_at_k_hybrid': round(float(np.mean(hybrid_overlap)), 4),
            'text_hit_rate_at_k_image_only': round(float(hit_rate_image), 4),
            'text_hit_rate_at_k_hybrid': round(float(hit_rate_hybrid), 4),
            'hybrid_gain_hit_rate': round(float(hit_rate_hybrid - hit_rate_image), 4),
            'mrr_at_k_image_only': round(float(np.mean(image_rr)), 4),
            'mrr_at_k_hybrid': round(float(np.mean(hybrid_rr)), 4),
            'hybrid_gain_mrr': round(float(np.mean(hybrid_rr) - np.mean(image_rr)), 4),
            'ms_per_query_image': round(float(dt_img / len(Qi)), 4),
            'ms_per_query_text': round(float(dt_txt / len(Qt)), 4),
            'ms_per_query_image_on_text': round(float(dt_img_on_text / len(Qt)), 4),
            'queries_image': len(Qi),
            'queries_text': len(Qt),
            'k': K,
        })

    version = save_version(
        image_idx,
        image_ids=image_ids,
        kind=kind,
        build_params=build_params,
        text_idx=text_idx,
        text_image_ids=text_image_ids,
    )
    saved.append({
        'version': version,
        'kind': kind,
        'build_params': build_params,
        'has_text_index': True,
    })

results_df = pd.DataFrame(eval_rows).sort_values(
    ['hybrid_gain_hit_rate', 'text_hit_rate_at_k_hybrid', 'ms_per_query_image'],
    ascending=[False, False, True],
)
display(results_df)

print('saved versions:')
display(pd.DataFrame(saved))

print('all versions:')
display(list_versions())


In [ ]:
# ==== Query comparison: always show image-only vs hybrid side-by-side ====
# Query modes:
# - 'image_id': use an existing image's embedding as query
# - 'vector_file': load query vector from local .npy file
# - 'text': encode text query locally with SigLIP (transformers)
QUERY_MODE = 'text'
QUERY_IMAGE_ID = int(meta_df.iloc[0]['image_id'])
QUERY_VECTOR_FILE = None  # e.g. '/tmp/query_vec.npy'
TEXT_QUERY = 'drake meme'
TOP_K = 12

# Which versions to compare:
# - None => latest per kind
# - list[str] => exact versions
SELECTED_VERSIONS = None

# Per-kind search params used during search
SEARCH_PARAMS_BY_KIND = {
    'flat': {},
    'ivf': {'nprobe': 32},
    'hnsw': {'efSearch': 128},
    'ivfpq': {'nprobe': 32},
}

RRF_K = 60

_TEXT_ENCODER = None


def get_text_encoder():
    global _TEXT_ENCODER
    if _TEXT_ENCODER is not None:
        return _TEXT_ENCODER

    import torch
    from transformers import AutoModel, AutoProcessor

    model_name = EMBED_MODEL
    device = 'cuda' if EMBED_DEVICE == 'cuda' and torch.cuda.is_available() else 'cpu'

    processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
    model = model.to(device)
    model.eval()

    is_siglip2 = 'siglip2' in model_name.lower()
    has_get_text_features = hasattr(model, 'get_text_features')

    def encode(text: str) -> np.ndarray:
        t = text.lower() if is_siglip2 else text
        kwargs = {'text': [t], 'return_tensors': 'pt', 'padding': 'max_length'}
        if is_siglip2:
            kwargs['max_length'] = 64
        inputs = processor(**kwargs)

        moved = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            if has_get_text_features:
                feats = model.get_text_features(**moved)
            else:
                out = model(**moved)
                if hasattr(out, 'text_embeds'):
                    feats = out.text_embeds
                elif hasattr(out, 'pooler_output'):
                    feats = out.pooler_output
                elif hasattr(out, 'last_hidden_state'):
                    feats = out.last_hidden_state[:, 0, :]
                else:
                    raise RuntimeError('Unknown model output format for text features')

        v = feats.detach().cpu().numpy().astype(np.float32)
        faiss.normalize_L2(v)
        return v

    _TEXT_ENCODER = encode
    return _TEXT_ENCODER


def choose_versions(df: pd.DataFrame, selected=None):
    if df.empty:
        return []
    if selected:
        allowed = set(df['version'])
        return [v for v in selected if v in allowed]

    chosen = []
    for kind in ['flat', 'ivf', 'hnsw', 'ivfpq']:
        sub = df[df['kind'] == kind]
        if not sub.empty:
            chosen.append(sub.iloc[0]['version'])
    return chosen


def get_query_vector() -> np.ndarray:
    if QUERY_MODE == 'image_id':
        id_to_pos = {int(img_id): i for i, img_id in enumerate(meta_df['image_id'].astype(int).tolist())}
        if QUERY_IMAGE_ID not in id_to_pos:
            raise ValueError(f'QUERY_IMAGE_ID not found in loaded data: {QUERY_IMAGE_ID}')
        return Xn[id_to_pos[QUERY_IMAGE_ID]].reshape(1, -1)

    if QUERY_MODE == 'vector_file':
        if not QUERY_VECTOR_FILE:
            raise ValueError('Set QUERY_VECTOR_FILE when QUERY_MODE=vector_file')
        v = np.load(QUERY_VECTOR_FILE).astype(np.float32).reshape(1, -1)
        faiss.normalize_L2(v)
        return v

    if QUERY_MODE == 'text':
        if not TEXT_QUERY or not TEXT_QUERY.strip():
            raise ValueError('Set TEXT_QUERY when QUERY_MODE=text')
        encode = get_text_encoder()
        return encode(TEXT_QUERY)

    raise ValueError('Unknown QUERY_MODE')


def preview_from_s3(df: pd.DataFrame, title: str, cols: int = 4):
    if df.empty:
        print(f'{title}: no hits')
        return

    rows_n = int(np.ceil(len(df) / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(4.8 * cols, 4.2 * rows_n))
    fig.suptitle(title)
    axes = np.array(axes).reshape(-1)

    for i, ax in enumerate(axes):
        if i >= len(df):
            ax.axis('off')
            continue

        r = df.iloc[i]
        key = r.get('s3_key')
        try:
            if not key:
                raise RuntimeError('missing s3_key')
            obj = s3.get_object(Bucket=S3_BUCKET, Key=key)
            img = Image.open(io.BytesIO(obj['Body'].read())).convert('RGB')
            ax.imshow(img)
        except Exception as exc:
            ax.text(0.5, 0.5, f'preview failed\n{exc}', ha='center', va='center')

        ax.set_title(f"id={int(r['image_id'])} score={float(r['score']):.4f}")
        ax.axis('off')

    plt.tight_layout()
    plt.show()


def to_hits_df(results, version, kind, mode_label, latency_ms):
    rows = []
    for img_id, score in results:
        if QUERY_MODE == 'image_id' and img_id == QUERY_IMAGE_ID:
            continue

        hit = meta_df.loc[meta_df['image_id'] == img_id].head(1)
        if hit.empty:
            continue

        rec = hit.iloc[0].to_dict()
        rec['score'] = float(score)
        rec['version'] = version
        rec['kind'] = kind
        rec['mode'] = mode_label
        rec['latency_ms'] = round(latency_ms, 3)
        rows.append(rec)

        if len(rows) >= TOP_K:
            break

    return pd.DataFrame(rows)


versions_df = list_versions()
if versions_df.empty:
    raise RuntimeError('No saved versions. Run build/eval cell first.')

target_versions = choose_versions(versions_df, SELECTED_VERSIONS)
if not target_versions:
    raise RuntimeError('No versions selected to compare.')

qvec = get_query_vector()
if qvec.shape[1] != Xn.shape[1]:
    raise ValueError(f'Query dim {qvec.shape[1]} does not match index dim {Xn.shape[1]}. Use same EMBED_MODEL as indexed embeddings.')

summary_rows = []

for version in target_versions:
    idx, mapping, idx_meta, text_idx, text_mapping, _ = load_version(version)
    kind = idx_meta.get('kind', 'unknown')
    params = SEARCH_PARAMS_BY_KIND.get(kind, {})
    set_search_params(idx, **params)
    if text_idx is not None:
        set_search_params(text_idx, **params)

    t0 = time.perf_counter()
    image_results = search_faiss(idx, mapping, qvec, TOP_K + 1)
    image_ms = (time.perf_counter() - t0) * 1000.0

    hybrid_results = image_results
    hybrid_ms = image_ms
    if text_idx is not None:
        t1 = time.perf_counter()
        text_results = search_faiss(text_idx, text_mapping, qvec, TOP_K + 1)
        text_ms = (time.perf_counter() - t1) * 1000.0
        hybrid_results = reciprocal_rank_fusion(image_results, text_results, k=RRF_K)
        hybrid_ms = image_ms + text_ms

    image_df = to_hits_df(image_results, version, kind, 'image_only', image_ms)
    hybrid_df = to_hits_df(hybrid_results, version, kind, 'hybrid', hybrid_ms)

    image_ids_set = set(image_df['image_id'].astype(int).tolist()) if not image_df.empty else set()
    hybrid_ids_set = set(hybrid_df['image_id'].astype(int).tolist()) if not hybrid_df.empty else set()

    overlap = len(image_ids_set & hybrid_ids_set)
    denom = max(1, min(len(image_ids_set), len(hybrid_ids_set)))
    overlap_at_k = overlap / denom

    hybrid_only = sorted(hybrid_ids_set - image_ids_set)
    image_only = sorted(image_ids_set - hybrid_ids_set)

    summary_rows.append({
        'version': version,
        'kind': kind,
        'k': TOP_K,
        'image_latency_ms': round(image_ms, 3),
        'hybrid_latency_ms': round(hybrid_ms, 3),
        'overlap_at_k': round(overlap_at_k, 4),
        'hybrid_new_ids': len(hybrid_only),
        'image_only_ids': len(image_only),
        'has_text_index': text_idx is not None,
    })

    print(f'\n=== {version} ({kind}) ===')
    print({'hybrid_new_ids': len(hybrid_only), 'image_only_ids': len(image_only), 'overlap_at_k': round(overlap_at_k, 4)})

    if not image_df.empty:
        print('image-only top hits:')
        display(image_df[['version', 'kind', 'mode', 'latency_ms', 'image_id', 'score', 'dataset', 'caption']])
        preview_from_s3(image_df, title=f'{version} ({kind}) [image-only]', cols=4)

    if not hybrid_df.empty:
        print('hybrid top hits:')
        display(hybrid_df[['version', 'kind', 'mode', 'latency_ms', 'image_id', 'score', 'dataset', 'caption']])
        preview_from_s3(hybrid_df, title=f'{version} ({kind}) [hybrid]', cols=4)

print('\nsummary across versions:')
display(pd.DataFrame(summary_rows).sort_values(['hybrid_new_ids', 'overlap_at_k'], ascending=[False, True]))
